# Notebook 1b — Concept Vocabulary

Splits all SNOMED breast concept preferred terms on spaces and builds a deduplicated word-level dictionary.

**Input**
- `data/embeddings-concept-openai/concepts.csv` — breast concept preferred terms

**Outputs**
- `data/stage1/concept_vocabulary.pkl` — `{word: [concept_id, ...]}` bare words
- `data/stage1/concept_vocabulary_spaced.pkl` — `{" " + word: [concept_id, ...]}` space-prefixed words

In BPE tokenizers (Llama-3, GPT-2 family), a word mid-sentence is encoded with the leading space baked into the token — `"breast"` and `" breast"` have different token IDs. The spaced form is what appears inside phrases and is what the footprints probes need to look up the correct token ID.

**Motivation**

SNOMED concepts are multi-word phrases, not single multi-token words. The footprints probes (notebook 1) detect subword-token merging within a *single word* — they cannot be applied meaningfully to a full phrase like "invasive ductal carcinoma of breast". This notebook extracts the 1,490 individual words across all 1,879 concept preferred terms so that downstream calibration can operate on words rather than phrases.

In [ ]:
# parameters
DATA_DIR     = "../../data/stage1"
CONCEPTS_CSV = "../../data/embeddings-concept-openai/concepts.csv"

In [ ]:
import os
import pickle

import pandas as pd

os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(CONCEPTS_CSV)
print(f"Concepts loaded: {len(df)}")
import sys
import time
import psutil as _psutil
from datetime import datetime

def _ts():
    return datetime.now().strftime("%H:%M:%S")


In [ ]:
import subprocess as _sub
_nb_start = time.time()
try:
    _git_hash = _sub.check_output(["git","rev-parse","--short","HEAD"], stderr=_sub.DEVNULL).decode().strip()
except Exception:
    _git_hash = "unknown"
print(f"[{_ts()}] \u2550" * 42 + "\u2550")
print(f"[{_ts()}]  Notebook 1b \u2014 Concept Vocabulary")
print(f"[{_ts()}]  git: {_git_hash}   DATA_DIR: {DATA_DIR}")
print(f"[{_ts()}]  RAM: {_psutil.virtual_memory().total/1e9:.1f}GB total  {_psutil.virtual_memory().available/1e9:.1f}GB avail")
print(f"[{_ts()}] \u2550" * 42 + "\u2550")
sys.stdout.flush()

In [ ]:
_cell_start = time.time()
print(f"[{_ts()}] === Build concept vocabulary ==="); sys.stdout.flush()
vocab = {}
for _i, (_, row) in enumerate(df.iterrows()):
    if _i % 5000 == 0:
        print(f"[{_ts()}]  row {_i}/{len(df)}  elapsed {time.time()-_cell_start:.1f}s"); sys.stdout.flush()
    cid = str(row["concept_id"])
    for word in str(row["preferred_term"]).split(" "):
        word = word.strip()
        if word and word.isalpha() and word.isascii():
            if word not in vocab:
                vocab[word] = []
            if cid not in vocab[word]:
                vocab[word].append(cid)

for word in vocab:
    vocab[word].sort()

print(f"Unique words: {len(vocab)}")
print(f"Sample entries:")
for word in list(sorted(vocab))[:5]:
    print(f"  {word!r}: {len(vocab[word])} concept(s)")
_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Cell complete  elapsed {time.time()-_cell_start:.1f}s{_rss_end}"); sys.stdout.flush()

In [ ]:
import shutil as _shutil
_free_gb = _shutil.disk_usage(DATA_DIR).free / 1e9
print(f"[{_ts()}]  Disk free: {_free_gb:.1f}GB"); sys.stdout.flush()
out_path = os.path.join(DATA_DIR, "concept_vocabulary.pkl")
with open(out_path, "wb") as f:
    pickle.dump(vocab, f)
import os as _os2
_sz_mb = _os2.path.getsize(out_path) / 1e6
print(f"Saved: {out_path}  ({_sz_mb:.1f} MB)")

In [ ]:
vocab_spaced = {" " + word: cids for word, cids in vocab.items()}

_free_gb = _shutil.disk_usage(DATA_DIR).free / 1e9
print(f"[{_ts()}]  Disk free: {_free_gb:.1f}GB"); sys.stdout.flush()
out_path_spaced = os.path.join(DATA_DIR, "concept_vocabulary_spaced.pkl")
with open(out_path_spaced, "wb") as f:
    pickle.dump(vocab_spaced, f)
_sz_mb = _os2.path.getsize(out_path_spaced) / 1e6
print(f"Saved: {out_path_spaced}  ({_sz_mb:.1f} MB)")
_rss_end = f"  RAM: {_psutil.Process().memory_info().rss/1e9:.2f}GB RSS"
print(f"[{_ts()}]  Notebook complete — total {time.time()-_nb_start:.0f}s elapsed{_rss_end}"); sys.stdout.flush()